# Data preparation for fine-tuning

This notebook covers:
1. Installing dependencies
2. Fetching and preparing the ISS incident evaluation dataset
3. Evaluating the teacher model (gpt-4.1-mini via APIM)
4. Generating synthetic training data via knowledge distillation

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parents[0]
load_dotenv(repo_root / '.env', override=True)

In [ ]:
# Setup Dependencies
%pip install openai azure-ai-inference azure-identity pandas matplotlib transformers peft torch tqdm -q
print("Dependencies installed.")

In [ ]:
import os
import re
import json
import torch
import pandas as pd
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

# Load local helpers
from iss_utils import get_evaluation_dataset, fetch_report, create_classification_prompt, parse_classification_response, evaluate_classification
from azure_infra import provision_infrastructure, submit_finetune_job, monitor_job, download_model

# Suppress tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Load environment
load_dotenv("../.env")
print("Environment loaded.")

## Prepare Evaluation Dataset

We will compile a dataset of real days from the ISS mission history-some with documented incidents (Critical/Warning) and some with routine operations (Nominal).

In [ ]:
# 1. Get List of Dates to Evaluate
eval_dataset = get_evaluation_dataset()
print(f"Dataset: {len(eval_dataset)} evaluation points")
print(f"- Incidents: {sum(1 for d in eval_dataset if d['is_incident'])}")
print(f"- Normal Days: {sum(1 for d in eval_dataset if not d['is_incident'])}")

# 2. Fetch NASA Reports for these days
print("\nFetching NASA reports...")
reports_data = {}
for item in tqdm(eval_dataset):
    date = item["date"]
    if date not in reports_data:
        res = fetch_report(date)
        if res["success"]:
            reports_data[date] = res

print(f"Successfully fetched {len(reports_data)} reports.")

In [ ]:
sample_date = list(reports_data.keys())[0]
sample_report = reports_data[sample_date]

clean_text = lambda t: re.sub(r'\n\s*\n+', '\n\n', re.sub(r':\s*\n+', ': ', t)).strip()
cleaned_text = clean_text(sample_report["report_text"])

print(f"Sample NASA Report for {sample_date}")
print("=" * 60)
print(cleaned_text[:1500])
print("\n... [truncated for display]" if len(cleaned_text) > 1500 else "")

## Evaluate Teacher Model (gpt-4.1-mini (via APIM gateway))

We use **gpt-4.1-mini (via APIM gateway)** (hosted on Azure APIM) as our "Teacher". It provides high-quality classifications that we use to:
1. Establish the target accuracy (~80%)
2. Generate training labels for knowledge distillation

> **Why DeepSeek?** As an open-source model, DeepSeek's outputs can be freely used for training derivative models-an important consideration when building knowledge distillation pipelines.

In [ ]:
# Configure Teacher Model
import os

GATEWAY_URL = os.getenv("GATEWAY_URL")
FINETUNE_GATEWAY_KEY = os.getenv("FINETUNE_GATEWAY_KEY")
TEACHER_MODEL = os.getenv("CHAT_MODEL", "gpt-4.1-mini")

client = AzureOpenAI(
    azure_endpoint=GATEWAY_URL.rstrip('/').replace('/openai', ''),
    api_key=FINETUNE_GATEWAY_KEY,
    api_version="2024-12-01-preview"
)
print(f"Teacher model: {TEACHER_MODEL}")
print(f"Gateway URL: {GATEWAY_URL}")

In [ ]:
# Sample Input/Output to DeepSeek (Teacher Model)
sample_date = list(reports_data.keys())[0]
sample_report_text = reports_data[sample_date]["report_text"]
sample_prompts = create_classification_prompt(sample_report_text)

print("INPUT TO DEEPSEEK")
print("=" * 60)
print("\n[SYSTEM PROMPT]")
print("-" * 40)
print(sample_prompts["system"][:800])
print("\n[USER PROMPT]")
print("-" * 40)
print(clean_text(sample_prompts["user"])[:1000])
print("\n... [truncated for display]" if len(sample_prompts["user"]) > 1000 else "")

# Query teacher for sample output
print("\n" + "=" * 60)
print("OUTPUT FROM DEEPSEEK")
print("=" * 60)
sample_response = client.chat.completions.create(
    model=TEACHER_MODEL,
    messages=[{"role": "system", "content": sample_prompts["system"]}, 
              {"role": "user", "content": sample_prompts["user"]}],
    temperature=0.1, max_tokens=500
)
print(sample_response.choices[0].message.content)

In [ ]:
def query_teacher(report_text):
    prompts = create_classification_prompt(report_text)
    response = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[{"role": "system", "content": prompts["system"]}, 
                  {"role": "user", "content": prompts["user"]}],
        temperature=0.1, max_tokens=500
    )
    return response.choices[0].message.content, prompts

# Run Evaluation
teacher_results = []
print("Evaluating Teacher (DeepSeek)...")

for item in tqdm(eval_dataset):
    if item["date"] not in reports_data: continue
    
    report = reports_data[item["date"]]
    raw_response, prompts = query_teacher(report["report_text"])
    
    # Parse & Grade
    prediction = parse_classification_response(raw_response)
    metrics = evaluate_classification(prediction, item)
    
    teacher_results.append({
        "date": item["date"],
        "metrics": metrics,
        "prediction": prediction,
        "raw_response": raw_response,
        "prompts": prompts  # Save for training data generation
    })

accuracy = sum(1 for r in teacher_results if r["metrics"]["severity_exact_match"]) / len(teacher_results)
print(f"\n🎓 Teacher Accuracy: {accuracy:.1%}")

## Evaluate Base model

Reference baseline from earlier local testing

In [ ]:
# Base model reference
# Phi-4-mini is available via the APIM gateway as 'Phi-4-mini-instruct'
# Use the reference baseline accuracy from earlier testing
BASE_MODEL_ID = "microsoft/Phi-4-mini-instruct"
BASE_MODEL_ENDPOINT = os.getenv("GATEWAY_URL")
base_acc = 0.457

print(f"Base Model: {BASE_MODEL_ID}")
print(f"Base Model Endpoint: {BASE_MODEL_ENDPOINT}")
print(f"Reference Baseline Accuracy: {base_acc:.1%}")
print("(Using reference value from earlier ACA evaluation - no local download required)")

## Generate Training Data (~500 Examples)

We create a high-quality training dataset by:
1. **Filtering Real Data**: Only use reports where the teacher's prediction matched ground truth
2. **Generating Synthetic Data**: Use DeepSeek to create realistic ISS reports with known severities

This gives us enough diverse examples for effective fine-tuning.

In [ ]:
# Load existing training data or generate new
TRAIN_FILE = "data/train.jsonl"

if os.path.exists(TRAIN_FILE):
    print(f"Loading existing training data from {TRAIN_FILE}...")
    all_training = []
    with open(TRAIN_FILE, "r") as f:
        for line in f:
            all_training.append(json.loads(line))
    print(f"Loaded {len(all_training)} examples")
    
    # Show severity distribution
    severities = []
    for ex in all_training:
        match = re.search(r'SEVERITY:\s*(\w+)', ex["assistant"], re.IGNORECASE)
        if match:
            severities.append(match.group(1).lower())
    print(f"Severity distribution: {Counter(severities)}")
else:
    print("No existing training data found. Generating...")
    
    # Reload iss_utils to pick up new functions
    import importlib
    import iss_utils
    importlib.reload(iss_utils)
    from iss_utils import get_synthetic_scenarios, create_synthetic_report_prompt, create_classification_prompt

    # --- Step 1: Filter Real Data (only correct predictions) ---
    real_training = []
    for res in teacher_results:
        if res["metrics"]["severity_exact_match"]:  # Only correct predictions
            example = {
                "system": res["prompts"]["system"],
                "user": res["prompts"]["user"],
                "assistant": res["raw_response"]
            }
            real_training.append(example)
            
    print(f"Real data (filtered): {len(real_training)} examples")

    # --- Step 2: Generate Synthetic Reports ---
    TARGET_TOTAL = 500
    synthetic_needed = TARGET_TOTAL - len(real_training)
    scenarios = get_synthetic_scenarios(synthetic_needed)

    print(f"Generating {len(scenarios)} synthetic reports...")
    synthetic_training = []

    for i, scenario in enumerate(tqdm(scenarios)):
        try:
            # Step 2a: Generate synthetic report
            gen_prompt = create_synthetic_report_prompt(scenario)
            gen_response = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {"role": "system", "content": gen_prompt["system"]},
                    {"role": "user", "content": gen_prompt["user"]}
                ],
                temperature=0.8,  # More creativity for variety
                max_tokens=800
            )
            synthetic_report = gen_response.choices[0].message.content
            
            # Step 2b: Have teacher classify the synthetic report
            class_prompt = create_classification_prompt(synthetic_report)
            class_response = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {"role": "system", "content": class_prompt["system"]},
                    {"role": "user", "content": class_prompt["user"]}
                ],
                temperature=0.1,
                max_tokens=500
            )
            classification = class_response.choices[0].message.content
            
            # Create training example
            example = {
                "system": class_prompt["system"],
                "user": class_prompt["user"],
                "assistant": classification
            }
            synthetic_training.append(example)
            
        except Exception as e:
            print(f"Error on scenario {i}: {e}")
            continue

    print(f"Synthetic data: {len(synthetic_training)} examples")

    # --- Step 3: Combine and Save ---
    all_training = real_training + synthetic_training
    print(f"\nTotal training examples: {len(all_training)}")

    # Severity distribution check
    severities = []
    for ex in all_training:
        match = re.search(r'SEVERITY:\s*(\w+)', ex["assistant"], re.IGNORECASE)
        if match:
            severities.append(match.group(1).lower())
    print(f"Severity distribution: {Counter(severities)}")

    # Save to JSONL
    os.makedirs("data", exist_ok=True)
    with open(TRAIN_FILE, "w") as f:
        for ex in all_training:
            f.write(json.dumps(ex) + "\n")

    print(f"\nSaved {len(all_training)} examples to {TRAIN_FILE}")

In [ ]:
# Sample Training Example (for Fine-Tuning)
print("SAMPLE TRAINING EXAMPLE")
print("=" * 60)

sample_example = all_training[0]

print("\n[SYSTEM]")
print("-" * 40)
print(sample_example["system"][:600])

print("\n[USER] (Input)")
print("-" * 40)
print(clean_text(sample_example["user"])[:800])
print("... [truncated]" if len(sample_example["user"]) > 800 else "")

print("\n[ASSISTANT] (Expected Output)")
print("-" * 40)
print(sample_example["assistant"])

print("\n" + "=" * 60)
print(f"Total training examples: {len(all_training)}")
print(f"Format: Each example has 'system', 'user', and 'assistant' fields")